# CT-CLIP feature extraction (CT-RATE-native encoder) — Colab

Re-encode the CT-RATE longitudinal **600-pair subset (≈ 1,114 unique volumes)** with the
chest-native **CT-CLIP** model instead of MERLIN, to feed the global TempA-VLP
static+dynamic temporal model.

**Safety-first design:**
- Outputs are written **directly to Google Drive, one 512-d file per volume, as it is computed** (continuous auto-save).
- The loop **skips already-cached** volumes → if a Colab session dies, just re-run this notebook and it **resumes** where it stopped.
- Volumes are **streamed one at a time and deleted immediately** → peak disk < 1 GB (works on tiny Colab/local storage).

Worst case on a disconnect: you lose the *single* volume in flight (a few seconds).

## 1. GPU check

In [ ]:
!nvidia-smi || echo 'NO GPU — set Runtime > Change runtime type > GPU (A100/L4 ideal, >=24 GB)'
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 2. Setup: clone CT-CLIP + install deps
CTViT is picky about torch — if imports fail, restart runtime after this cell once.

In [ ]:
%cd /content
![ -d CT-CLIP ] || git clone https://github.com/ibrahimethemhamamci/CT-CLIP.git
%cd /content/CT-CLIP
!pip install -e transformer_maskgit
!pip install -e CT_CLIP
!pip install -q nibabel scipy huggingface_hub transformers tqdm
%cd /content

# Fallback: import packages straight from the cloned source, in case the
# editable-install registration didn't take in this kernel.
import sys
for p in ['/content/CT-CLIP/CT_CLIP', '/content/CT-CLIP/transformer_maskgit']:
    if p not in sys.path:
        sys.path.insert(0, p)
import ct_clip, transformer_maskgit
print('ct_clip           ->', ct_clip.__file__)
print('transformer_maskgit ->', transformer_maskgit.__file__)

## 3. Mount Drive + get our repo helper (`ctclip_utils.py`)
Set `DRIVE_OUT` to where the tiny embedding cache should persist.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# --- edit these two if your paths differ ---
DRIVE_OUT   = '/content/drive/MyDrive/3dCT/ctclip_cache'   # embeddings persist here (auto-saved)
REPO_URL    = 'https://github.com/nprakash1/3dCT.git'    # for subset_pairs.csv + labels + ctclip_utils.py
# -------------------------------------------
os.makedirs(DRIVE_OUT, exist_ok=True)
os.makedirs(os.path.join(DRIVE_OUT, 'img'),  exist_ok=True)
os.makedirs(os.path.join(DRIVE_OUT, 'text'), exist_ok=True)

%cd /content
![ -d 3dCT ] || git clone $REPO_URL 3dCT
import sys; sys.path.append('/content/3dCT/scripts')
print('cache ->', DRIVE_OUT)

## 4. HuggingFace auth (CT-RATE + CT-CLIP are gated)

In [ ]:
from huggingface_hub import login
login()  # paste a READ token; accept CT-RATE + (if needed) CT-CLIP licenses on HF first
import os; os.environ.setdefault('HF_TOKEN', '')  # optionally set here

## 5. Download CT-CLIP weights (once) — cache to Drive so sessions reuse it

In [ ]:
from huggingface_hub import hf_hub_download
from ctclip_utils import REPO_ID, CTCLIP_WEIGHTS_HF

WEIGHTS_DIR = '/content/drive/MyDrive/3dCT/ctclip_weights'
os.makedirs(WEIGHTS_DIR, exist_ok=True)
weights_path = hf_hub_download(REPO_ID, CTCLIP_WEIGHTS_HF, repo_type='dataset',
                               token=os.environ.get('HF_TOKEN') or True,
                               local_dir=WEIGHTS_DIR)
print('weights ->', weights_path)

## 6. Build the frozen CT-CLIP embedder

In [ ]:
from ctclip_utils import CTCLIPEmbedder
emb = CTCLIPEmbedder(weights_path)
print('device:', emb.device)

## 6b. VERIFY model API (catch attribute/shape drift before the full run)
If any of `visual_transformer / text_transformer / to_visual_latent / to_text_latent`
is missing, print the real attribute names and adjust `ctclip_utils.py` accordingly.

In [ ]:
for a in ['visual_transformer','text_transformer','to_visual_latent','to_text_latent']:
    print(a, '->', hasattr(emb.clip, a))
print('\nattrs:', [a for a in dir(emb.clip) if not a.startswith('_')][:40])
# quick text-shape check
t = emb.embed_texts(['pleural effusion has increased', 'stable nodule'])
print('text emb shape:', tuple(t.shape))  # expect (2, 512)

## 7. Separability spike (the key free result)
Re-run the direction-blindness diagnostic on **CXR-BERT** (CT-CLIP's text tower).
Compare to MERLIN's Clinical-Longformer numbers (worsened/stable/improved ≈ 0.93,
worsened↔improved ≈ 0.94). Higher separation here = the chest-native encoder is less
direction-blind (a slide on its own).

In [ ]:
import itertools, torch
findings = ['pleural effusion','pulmonary nodule','consolidation','cardiomegaly','atelectasis']
changes  = {'worsened':'{} has increased','stable':'{} is unchanged','improved':'{} has decreased'}
prompts, meta = [], []
for f in findings:
    for c, tmpl in changes.items():
        prompts.append(tmpl.format(f)); meta.append((f, c))
E = emb.embed_texts(prompts, normalize=True)
cos = E @ E.T
def avg(pred):
    vals=[cos[i,j].item() for i in range(len(meta)) for j in range(len(meta)) if i<j and pred(meta[i],meta[j])]
    return sum(vals)/len(vals) if vals else float('nan')
print('same finding, diff change :', round(avg(lambda a,b: a[0]==b[0] and a[1]!=b[1]),3))
print('diff finding, same change :', round(avg(lambda a,b: a[0]!=b[0] and a[1]==b[1]),3))
print('worsened<->improved       :', round(avg(lambda a,b: {a[1],b[1]}=={'worsened','improved'} and a[0]==b[0]),3))

## 8. Load the 600-pair subset manifest
Uses `subset_pairs.csv` if present in the repo; otherwise falls back to the labeled
pairs file. Builds the list of **unique volumes** to encode.

In [ ]:
import csv, json, glob
SUBSET_CSV = '/content/3dCT/data/ctrate/subset_pairs.csv'
pairs = []
if os.path.exists(SUBSET_CSV):
    with open(SUBSET_CSV) as f:
        pairs = list(csv.DictReader(f))
    print('loaded subset_pairs.csv:', len(pairs), 'pairs')
else:
    # fallback: derive from labels jsonl in the repo
    lab = glob.glob('/content/3dCT/**/medgemma_labels_v3.jsonl', recursive=True)[0]
    for line in open(lab):
        d = json.loads(line)
        if d.get('parse_ok'):
            pairs.append({'patient': d['patient'], 'prior_volume': d['prior_volume'],
                          'curr_volume': d['curr_volume'], 'delta_days': d['delta_days']})
    print('fallback from labels:', len(pairs), 'pairs (consider generating subset_pairs.csv)')

vols = []
for p in pairs:
    vols += [p['prior_volume'], p['curr_volume']]
vols = list(dict.fromkeys(vols))
print('unique volumes to encode:', len(vols))

## 9. Stream → encode → auto-save to Drive → delete  (RESUMABLE)
Each volume: download one file → CT-CLIP preprocess+forward → `torch.save` 512-d to
Drive → delete the volume. Re-running skips anything already in Drive.

In [ ]:
import time, json, torch
from tqdm.auto import tqdm
from ctclip_utils import download_volume

IMG_DIR = os.path.join(DRIVE_OUT, 'img')
TMP = '/content/_vol_tmp'; os.makedirs(TMP, exist_ok=True)
PROGRESS = os.path.join(DRIVE_OUT, '_progress.json')
token = os.environ.get('HF_TOKEN') or True

done, missing = 0, []
for i, v in enumerate(tqdm(vols)):
    out_pt = os.path.join(IMG_DIR, v.replace('.nii.gz','').replace('.nii','') + '.pt')
    if os.path.exists(out_pt):
        done += 1; continue                      # RESUME: already cached on Drive
    fp = download_volume(v, token, TMP)
    if not fp:
        missing.append(v); continue
    try:
        e = emb.embed_image_path(fp, normalize=True)      # (1, 512)
        torch.save(e.half().squeeze(0).clone(), out_pt)   # AUTO-SAVE to Drive now
        done += 1
    except Exception as ex:
        print('ENCODE FAIL', v, ex)
    finally:
        try: os.remove(fp)                                 # DELETE volume immediately
        except Exception: pass
    if (i+1) % 10 == 0:
        json.dump({'i': i+1, 'done': done, 'missing': len(missing), 't': time.time()},
                  open(PROGRESS,'w'))

json.dump({'i': len(vols), 'done': done, 'missing': len(missing)}, open(PROGRESS,'w'))
print(f'DONE image encode: {done} cached, {len(missing)} missing -> {IMG_DIR}')
if missing: print('missing (rerun to retry):', missing[:10])

## 10. Text embeddings: dynamic & static sentences per pair → Drive

In [ ]:
import glob, json, torch
TXT_DIR = os.path.join(DRIVE_OUT, 'text')
lab = glob.glob('/content/3dCT/**/medgemma_labels_v3.jsonl', recursive=True)
if lab:
    for line in tqdm(list(open(lab[0]))):
        d = json.loads(line)
        if not d.get('parse_ok'): continue
        key = d['curr_volume'].replace('.nii.gz','').replace('.nii','')
        out = os.path.join(TXT_DIR, key + '.pt')
        if os.path.exists(out): continue
        lab_obj = d.get('label') or d
        dyn = lab_obj.get('dynamic_sentences', []) or ['']
        sta = lab_obj.get('static_sentences', []) or ['']
        td = emb.embed_texts(dyn, normalize=True).mean(0)   # pooled dynamic (td)
        ts = emb.embed_texts(sta, normalize=True).mean(0)   # pooled static  (ts)
        torch.save({'td': td.half(), 'ts': ts.half(),
                    'n_dyn': len(dyn), 'n_sta': len(sta)}, out)
    print('text embeddings ->', TXT_DIR)
else:
    print('no labels jsonl found in repo clone; skip')

## 11. Verify: how many volumes cached vs expected

In [ ]:
import glob
n_img = len(glob.glob(os.path.join(DRIVE_OUT, 'img', '*.pt')))
n_txt = len(glob.glob(os.path.join(DRIVE_OUT, 'text', '*.pt')))
print(f'image embeddings cached: {n_img} / {len(vols)} expected')
print(f'text  embeddings cached: {n_txt}')
missing = [v for v in vols if not os.path.exists(os.path.join(DRIVE_OUT,'img',
           v.replace('.nii.gz','').replace('.nii','')+'.pt'))]
print('still missing:', len(missing), '(re-run cell 9 to fill)')